# HF-Bench Quickstart

> **⚠️ SYNTHETIC DATA ONLY** — All outputs in this notebook use a generated demo dataset and do NOT represent real clinical performance.

This notebook walks through the HF-Bench pipeline:
1. Generate demo data
2. Preprocess and split
3. Train a quick model
4. Evaluate metrics and fairness

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve, auc

print('HF-Bench quickstart — SYNTHETIC DATA ONLY')

## 1. Generate Demo Dataset

In [ ]:
from hfbench.data.demo import make_demo_dataset

df = make_demo_dataset(n=2000, seed=42)
print(f'Shape: {df.shape}')
print(f'Readmission rate: {df["readmit_30d"].mean():.1%}')
df.head()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
df['age'].hist(bins=30, ax=axes[0], color='steelblue')
axes[0].set_title('Age distribution')
df['readmit_30d'].value_counts().plot.bar(ax=axes[1], color=['#2196F3', '#F44336'])
axes[1].set_title('Label distribution (SYNTHETIC)')
axes[1].set_xticklabels(['No readmit', 'Readmit'], rotation=0)
df['length_of_stay'].hist(bins=30, ax=axes[2], color='seagreen')
axes[2].set_title('Length of stay (days)')
plt.tight_layout()
plt.show()

## 2. Temporal Split and Preprocessing

In [ ]:
from hfbench.data.split import temporal_split
from hfbench.data.preprocess import fit_preprocessor, transform, apply_smote
from hfbench.constants import LABEL_COL

df['age_group'] = (df['age'] >= 65).map({True: '>=65', False: '<65'})
train, val, test = temporal_split(df)
print(f'Train: {len(train)} | Val: {len(val)} | Test: {len(test)}')

preprocessor = fit_preprocessor(train)
X_train = transform(preprocessor, train)
y_train = train[LABEL_COL].values.astype(int)
X_val = transform(preprocessor, val)
y_val = val[LABEL_COL].values.astype(int)
X_test = transform(preprocessor, test)
y_test = test[LABEL_COL].values.astype(int)

print(f'Feature matrix shape: {X_train.shape}')
print(f'Train positive rate: {y_train.mean():.1%}')

In [ ]:
X_train_s, y_train_s = apply_smote(X_train, y_train, random_state=42)
print(f'After SMOTE: {X_train_s.shape}, positive rate: {y_train_s.mean():.1%}')

## 3. Train Models

In [ ]:
from hfbench.models.sklearn_models import LogisticRegressionModel

lr = LogisticRegressionModel(seed=42)
lr.fit(X_train_s, y_train_s)
lr_probs = lr.predict_proba(X_test)
print(f'LR probs range: [{lr_probs.min():.3f}, {lr_probs.max():.3f}]')

In [ ]:
try:
    from hfbench.models.xgb_model import XGBoostModel
    xgb = XGBoostModel(seed=42, n_estimators=50, max_depth=4)
    xgb.fit(X_train_s, y_train_s, X_val=X_val, y_val=y_val)
    xgb_probs = xgb.predict_proba(X_test)
    print('XGBoost trained successfully')
except ImportError:
    xgb_probs = None
    print('xgboost not installed, skipping')

## 4. Evaluate

In [ ]:
from hfbench.evaluation.metrics import compute_all_metrics

lr_metrics = compute_all_metrics(y_test, lr_probs)
print('=== Logistic Regression (SYNTHETIC DATA) ===')
for k in ['auroc', 'auprc', 'brier_score', 'ece']:
    print(f'  {k}: {lr_metrics[k]:.4f}')

In [ ]:
# ROC curves
fig, ax = plt.subplots(figsize=(6, 5))

fpr, tpr, _ = roc_curve(y_test, lr_probs)
ax.plot(fpr, tpr, label=f'LR (AUC={lr_metrics["auroc"]:.3f})')

if xgb_probs is not None:
    from hfbench.evaluation.metrics import compute_all_metrics as cam
    xgb_m = cam(y_test, xgb_probs)
    fpr2, tpr2, _ = roc_curve(y_test, xgb_probs)
    ax.plot(fpr2, tpr2, label=f'XGB (AUC={xgb_m["auroc"]:.3f})')

ax.plot([0,1],[0,1],'k--', alpha=0.4)
ax.set_xlabel('FPR')
ax.set_ylabel('TPR')
ax.set_title('ROC Curves — SYNTHETIC DATA ONLY')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Fairness Analysis

In [ ]:
from hfbench.evaluation.fairness import run_fairness_evaluation

meta = test[['sex', 'race', 'age_group']].reset_index(drop=True)
fairness = run_fairness_evaluation(y_test, lr_probs, meta, ['sex', 'age_group'])

for col, result in fairness.items():
    print(f'\n--- Subgroup: {col} ---')
    print(result['per_group'][['group','n','prevalence','auroc','tpr','fpr']].to_string(index=False))
    print('Summary:', result['summary'])

---
**Reminder**: All results above are from synthetic/demo data. They do NOT reflect real clinical model performance.